# 11 · The unsupervised detectors and the stack

**The questions.** DESIGN_QUESTIONS Part B, B7 and B6, which together answer RQ4: does combining supervised and unsupervised evidence help?
- **B7:** which unsupervised detectors to use. The current pipeline uses three (a peer-deviation score, an Isolation Forest, K-Means distance); B7 adds two candidates, a Gaussian mixture and DBSCAN. Local outlier factor and one-class SVM are left out because they do not scale to millions of records; the autoencoder was dropped with the MLP.
- **B6:** how the chosen supervised model (XGBoost, notebook 10) and the other members are combined into one score.

**Fixed before looking (B6, B7, with one change noted below):**
1. **Detectors** read the numeric deviation features (`ANOMALY_FEATURES_SHARED`), standardised, and are fitted on the training years 2000–2010 without labels. K-Means chooses its number of clusters by silhouette (with a stability check), the Gaussian mixture by BIC; DBSCAN takes the 95th percentile of the distance to the 10th neighbour as its radius, is fitted on a sample, and scores any record by its distance to the nearest core point.
2. **Each detector alone** is judged by PR-AUC on listed 2011 line items (expected weak: unusual is not illegal).
3. **One stacking layer, built by forward selection on 2011.** The pool holds the other five supervised candidates (notebook 10) and the five detectors. Start from XGBoost; add the member that most improves PR-AUC on listed line items; stop when the best addition's gain has a 95% paired-bootstrap interval (shipments resampled) that includes zero.
4. **How members are combined.** A logistic-regression meta-learner over the logit of each member's percentile rank. On 2011 it is cross-fitted: fitted on half of the shipments, scored on the other half, and the reverse.
5. **Then the combiner:** for the selected members, the meta-learner against an unweighted average of their percentile ranks (PRMLS Day 5). The average is kept if the two are tied.
6. If no detector enters the stack, that is RQ4's answer.

**The change.** B6 said members would be combined by averaging their ranks *during* selection, with the meta-learner compared only at the end. An unweighted average gives a weak detector the same weight as XGBoost, so it could never enter, whatever it would add with the right weight. Selection therefore uses the meta-learner, which B6 already names as the combiner to compare; the average is still compared at the end (step 5). The change was made before any stack result was seen.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

import eda_setup as E
import experiment_setup as X_
import model_selection as M
import stack_selection as SS
from wildguard import config
from wildguard.features import ANOMALY_FEATURES_SHARED as AF, FeatureBuilder
from wildguard.models import IsolationForestScorer, KMeansDistanceScorer, PeerDeviationScorer

E.style()
t0 = time.time()
df = X_.load_lemis()
cons = df.groupby("shipment_id")["source"].nunique().rename("cons_sources")
df = df[df["animal"]].join(cons, on="shipment_id").reset_index(drop=True)
tr, ev = df["year"].between(2000, 2010).to_numpy(), (df["year"] == 2011).to_numpy()
fb = FeatureBuilder(use_value=False).fit(df[tr])
F_tr, F_ev = fb.transform(df[tr]), fb.transform(df[ev])
S = pd.read_parquet(config.PROCESSED_DIR / "experiments" / "scores_2011.parquet")
assert (S["shipment_id"].to_numpy() == F_ev["shipment_id"].to_numpy()).all()
y, listed, ships = S["label"].to_numpy(), S["sp_listed"].to_numpy(), S["shipment_id"].to_numpy()
print(f"features ready: {len(F_tr):,} training, {len(F_ev):,} 2011 line items ({time.time() - t0:.0f}s)")

features ready: 3,562,032 training, 370,113 2011 line items (19s)


## 1. Fitting the detectors (B7)

In [2]:
detectors = {"peer_deviation": None,
             "isolation_forest": IsolationForestScorer(AF).fit(F_tr),
             "kmeans": KMeansDistanceScorer(AF).fit(F_tr),
             "gaussian_mixture": SS.GaussianMixtureScorer(AF).fit(F_tr),
             "dbscan": SS.DBSCANScorer(AF).fit(F_tr)}
D = {"peer_deviation": PeerDeviationScorer(use_value=False).score(F_ev)}
D.update({k: v.score(F_ev) for k, v in detectors.items() if v is not None})
km, gm, db = detectors["kmeans"], detectors["gaussian_mixture"], detectors["dbscan"]
print(f"K-Means: k = {km.k} by silhouette {km.silhouette}; stability (ARI) {km.stability_ari:.2f}")
print(f"Gaussian mixture: {gm.k} components by BIC {dict((k, round(v)) for k, v in gm.bic.items())}")
print(f"DBSCAN: radius {db.eps:.2f}, {db.n_clusters} clusters, {db.noise_share:.1%} of the sample noise")
print(f"({time.time() - t0:.0f}s)")

K-Means: k = 8 by silhouette {4: 0.28298207827214433, 6: 0.2731327022476633, 8: 0.29314074354988034, 10: 0.2930241382041745, 12: 0.2749465533891822}; stability (ARI) 0.83
Gaussian mixture: 12 components by BIC {2: 1294011, 4: 593117, 6: 513353, 8: 373786, 10: 340576, 12: 327288}
DBSCAN: radius 0.92, 8 clusters, 2.7% of the sample noise
(62s)


## 2. Each detector on its own

In [3]:
base_rate = y[listed].mean()
alone = pd.DataFrame({k: {"PR-AUC, listed": average_precision_score(y[listed], s[listed]),
                          "PR-AUC, all": average_precision_score(y, s)} for k, s in D.items()}).T
alone.loc["(random ranking)"] = [base_rate, y.mean()]
alone.style.format("{:.3f}")

,"PR-AUC, listed","PR-AUC, all"
peer_deviation,0.151,0.047
isolation_forest,0.111,0.071
kmeans,0.038,0.027
gaussian_mixture,0.086,0.056
dbscan,0.094,0.059
(random ranking),0.024,0.020


## 3. Forward selection of the stack (B6)

In [4]:
members = {c: S[c].to_numpy() for c in M.SIMPLICITY} | D
pool = [c for c in M.SIMPLICITY if c != "xgboost"] + list(D)
selected, steps = SS.forward_selection(members, y, ships, listed, "xgboost", pool,
                                       X_.pr_auc_bootstrap)
show = steps[["step", "current members", "best addition", "PR-AUC before", "PR-AUC with it",
              "gain", "95% low", "95% high", "added"]]
display(show.style.format({c: "{:.4f}" for c in ("PR-AUC before", "PR-AUC with it", "gain",
                                                 "95% low", "95% high")}))
print("Selected members:", selected)

step 1: best addition random_forest, gain 0.0098 [0.0059, 0.0135] -> added


step 2: best addition peer_deviation, gain 0.0007 [-0.0013, 0.0026] -> stop


,step,current members,best addition,PR-AUC before,PR-AUC with it,gain,95% low,95% high,added
0,1,xgboost,random_forest,0.5658,0.5756,0.0098,0.0059,0.0135,True
1,2,xgboost + random_forest,peer_deviation,0.5756,0.5763,0.0007,-0.0013,0.0026,False


Selected members: ['xgboost', 'random_forest']


What each pool member would have added at the first step (PR-AUC on listed 2011 line items with XGBoost plus that member):

In [5]:
first = steps.iloc[0]
pd.Series({p: first[f"PR-AUC with {p}"] for p in pool}).sort_values(ascending=False).to_frame(
    "PR-AUC, XGBoost + member").style.format("{:.4f}")

,"PR-AUC, XGBoost + member"
random_forest,0.5756
peer_deviation,0.5675
kmeans,0.5648
gaussian_mixture,0.5643
dbscan,0.5643
decision_tree,0.5643
isolation_forest,0.5642
logistic_regression,0.5618
adaboost,0.5617
naive_bayes,0.5599


For reference, not deciding: the current pipeline's stack (XGBoost with the peer-deviation score, the Isolation Forest and K-Means) on the same footing.

In [6]:
Z = {k: SS.logit_rank(v) for k, v in members.items()}
ref = {"XGBoost alone": SS.cross_fit(np.column_stack([Z["xgboost"]]), y, ships),
       "current stack": SS.cross_fit(np.column_stack([Z[k] for k in ("xgboost", "peer_deviation",
                                                                     "isolation_forest", "kmeans")]),
                                     y, ships)}
X_.pr_auc_bootstrap(y[listed], {k: v[listed] for k, v in ref.items()}, ships[listed],
                    [("current stack", "XGBoost alone")]).style.format(
    {c: "{:.4f}" for c in ("pr_auc_a", "pr_auc_b", "difference", "low_95", "high_95")})

,a,b,pr_auc_a,pr_auc_b,difference,low_95,high_95
0,current stack,XGBoost alone,0.5662,0.5658,0.0004,-0.0029,0.0035


## 4. The combiner

In [7]:
if len(selected) > 1:
    meta = SS.cross_fit(np.column_stack([Z[k] for k in selected]), y, ships)
    avg = np.mean([pd.Series(members[k]).rank(pct=True).to_numpy() for k in selected], axis=0)
    comb = X_.pr_auc_bootstrap(y[listed], {"meta-learner": meta[listed], "rank average": avg[listed]},
                               ships[listed], [("meta-learner", "rank average")])
    display(comb.style.format({c: "{:.4f}" for c in ("pr_auc_a", "pr_auc_b", "difference",
                                                     "low_95", "high_95")}))
    COMBINER = "rank average" if comb["low_95"].iat[0] <= 0 else "meta-learner"
else:
    COMBINER = "none: XGBoost alone"
print("Combiner:", COMBINER)

,a,b,pr_auc_a,pr_auc_b,difference,low_95,high_95
0,meta-learner,rank average,0.5756,0.5747,0.0008,-0.0001,0.0017


Combiner: rank average


## Findings

| # | Finding |
|---|---|
| H1 | **Detector settings found without labels:** K-Means 8 clusters (silhouette 0.29, stability 0.83); Gaussian mixture 12 components (lowest BIC); DBSCAN radius 0.92, 8 clusters, 2.7% of the sample as noise |
| H2 | **Each detector alone** (PR-AUC on listed 2011 line items; a random ranking scores 0.024): peer deviation 0.151, Isolation Forest 0.111, DBSCAN 0.094, Gaussian mixture 0.086, K-Means 0.038. All rank enforcement cases above random, all far below the supervised models (XGBoost 0.565): unusual is not illegal |
| H3 | **Forward selection:** random forest enters first (+0.0098, interval +0.006 to +0.014). The next-best addition, the peer-deviation score, adds +0.0007 (interval −0.001 to +0.003), so selection stops. **No detector enters the stack** |
| H4 | The current pipeline's stack (XGBoost with peer deviation, Isolation Forest and K-Means) is no better than XGBoost alone on the same footing (+0.0004, interval −0.003 to +0.004) |
| H5 | **Combiner:** for XGBoost and random forest, the meta-learner and an unweighted rank average are tied (+0.0008, interval −0.0001 to +0.0017), so the rank average is kept |
| H6 | **Answer to RQ4 on LEMIS:** combining supervised models (bagging and boosting) helps a little; adding unsupervised evidence to them does not |
| H7 | **Deployment cost of the forest** (tested after the selection, on the same 2011 footing): with 200 trees of depth 20 the ensemble gains +0.009 for a 96 MB model (~340 MB in memory); with 100 trees of depth 16 it gains +0.006 (interval +0.003 to +0.010) for 27 MB (~90 MB); with 50 trees of depth 12 the gain disappears (−0.001) |

**Decision (B6, B7, 31 October 2026):** the score is the average of the percentile ranks of XGBoost (notebook 10) and a random forest of 100 trees of depth 16, chosen over the 200-tree forest for the hosted app's memory (H7). No unsupervised detector is part of the score; the peer-deviation score is shown on the flag card as context for the reviewer. The other four detectors are reported as tested and rejected.